# **Module Overview**
This module covers everything you need to know about parsing and ingesting data for RAG systems, from basic text files to complex PDFs and databases. We'll use LangChain v0.3 and explore each technique with practical examples.

Table of contents

- Introduction to Data Ingestion
- Text Files (.txt)
- Markdown files (.md)
- PDF documents
- Microsoft Word Documents
- CSV and Excel Files
- JSON and Structured Data
- Web Scraping
- Databases (SQL)
- Audio and Video Transcripts
- Advanced Techniques
- Best Practices

## **Introduction to Data Ingestion**

In [1]:
import os
import pandas as pd

from typing import List, Dict, Any

In [2]:
from langchain_core.documents import Document
from langchain.document_loaders import TextLoader, DirectoryLoader
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

print("setup completed")

setup completed


## **Understanding Document Structure In Langchain**

In [3]:
# create a simple document
# page-content is where it's gonna hold all the details of actual document
# metadata is the field where we provide all other information related to that document, it can be anything
doc = Document(
    page_content="This is the main text content that will be embedded and searched",
    metadata={
        "source":"example.txt",
        "page":1,
        "author":"Dineshkumar Anbalagan",
        "data_created":"2026.03.08",
        "custom_field":"any_value"
        
    }
)

print("Document Structure")

print(f"Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")

# why metadata matters?
print("\n- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")

Document Structure
Content: This is the main text content that will be embedded and searched
Metadata: {'source': 'example.txt', 'page': 1, 'author': 'Dineshkumar Anbalagan', 'data_created': '2026.03.08', 'custom_field': 'any_value'}

- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


### **.txt Files - The Simplest Case {#2 - text-files}**

In [4]:
# create a simple txt file in data/text_files - did manually

### **TextLoader - Read Single File**

In [5]:
from langchain.document_loaders import TextLoader

loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")

documents = loader.load() # store each page as document object in list (multiple list elements in case of multiple pages)
print(f"Page loaded : {len(documents)} document")
print(f"Content Preview : {documents[0].page_content[:100]}...")
print(f"Metadata : {documents[0].metadata}")

print("\nreturn types")
print(f"TextLoader load() return type : {type(documents)}") # TextLoader Type
print(f"Type of element from list returned by TextLoader's load() : {type(documents[0])}") # Document Type

Page loaded : 1 document
Content Preview : What is Python?
Python is a high-level, interpreted programming language created by Guido van Rossum...
Metadata : {'source': 'data/text_files/python_intro.txt'}

return types
TextLoader load() return type : <class 'list'>
Type of element from list returned by TextLoader's load() : <class 'langchain_core.documents.base.Document'>


### **DirectoryLoader - Multiple Single File**

In [6]:
from langchain.document_loaders import TextLoader, DirectoryLoader

dir_loader=DirectoryLoader(
    path="data/text_files",
    glob="**/*.txt", ## pattern to match the files
    loader_cls=TextLoader, ## Loader class to use
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)

documents = dir_loader.load()
print(f"Loaded {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f"    source: {doc.metadata['source']}")
    print(f"    Length: {len(doc.page_content)} characters")

100%|██████████| 2/2 [00:00<00:00, 1147.55it/s]

Loaded 2 documents

Document 1:
    source: data/text_files/python_intro.txt
    Length: 2840 characters

Document 2:
    source: data/text_files/machine_learning.txt
    Length: 3082 characters


### **Text Splitting Strategies**

In [7]:
# different text splitting strategies
from langchain.text_splitter import (
    CharacterTextSplitter, 
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

In [8]:
print(documents, end="\n\n")
text = documents[0].page_content
print(text)

[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='What is Python?\nPython is a high-level, interpreted programming language created by Guido van Rossum in 1991. It emphasizes code readability and simplicity, making it accessible to beginners while remaining powerful for experienced developers. Python\'s design philosophy prioritizes writing clean, understandable code through features like indentation-based syntax instead of curly braces.\nAs an interpreted language, Python code executes line-by-line, enabling faster development and easier debugging. Its dynamic typing means variables don\'t require explicit type declarations, offering flexibility and reducing boilerplate. Python supports both object-oriented and functional programming paradigms, allowing developers to choose the approach that best fits their problem.\nPython\'s strength lies in its "batteries included" philosophy—a comprehensive standard library covering file I/O, system operations, mathe

In [108]:
### Method 1 - Character Text Splitter
char_splitter = CharacterTextSplitter(
    separator="\n", # split on new lines
    chunk_size=20, # Max chunk size in characters
    chunk_overlap=5, # Overlap between chunks
    length_function=len # How to measure chunk size
)

char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}..")

Created a chunk of size 373, which is longer than the specified 20
Created a chunk of size 384, which is longer than the specified 20
Created a chunk of size 1248, which is longer than the specified 20


Created 5 chunks
First chunk: What is Python?..


In [109]:
print(char_chunks[0])
print("------")
print(char_chunks[1])
print(len(char_chunks[1]))

What is Python?
------
Python is a high-level, interpreted programming language created by Guido van Rossum in 1991. It emphasizes code readability and simplicity, making it accessible to beginners while remaining powerful for experienced developers. Python's design philosophy prioritizes writing clean, understandable code through features like indentation-based syntax instead of curly braces.
373


In [ ]:
### Method 2 - Recursive Character Splitting (RECOMMENDED)
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=[" ", "\n"], # split on new lines
    chunk_size=60, # Max chunk size in characters
    chunk_overlap=10, # Overlap between chunks
    length_function=len # How to measure chunk size
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}..")

Created 56 chunks
First chunk: What is Python?
Python is a high-level, interpreted..


In [134]:
print(recursive_chunks[0])
print("------")
print(recursive_chunks[1])
print("------")
print(recursive_chunks[2])
print(len(recursive_chunks[1]))

What is Python?
Python is a high-level, interpreted
------
programming language created by Guido van Rossum in 1991.
------
in 1991. It emphasizes code readability and simplicity,
57


In [9]:
### Method 3 - Token based splitting
token_splitter = TokenTextSplitter(
    chunk_size=50, # size in token (not characters)
    chunk_overlap=10
)

token_chunks = token_splitter.split_text(text)
print(f"Created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}...")

Created 13 chunks
First chunk: What is Python?
Python is a high-level, interpreted programming language created by Guido van Rossum...
